# AI as Judge

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM as a judge to evaluate LLM outputs. The evaluation can be based on any criteria. G-Eval is implemented by a library called [DeepEval](https://deepeval.com/) which includes a broader set of tests.


In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from openai import OpenAI
import os

document_folder = "../../05_src/documents/"
blue_cross_file = "the_blue_cross.txt"
file_path = os.path.join(document_folder, blue_cross_file)

with open(file_path, "r", encoding="utf-8") as f:
    blue_cross_text = f.read()

In [3]:
instructions = "You are an helpful assistant that summarizes works of fiction with a quirky and bubbly approach."
PROMPT = """
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    {story}
    </story>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [4]:
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=1.2
)

In [5]:
from IPython.display import display, Markdown
import textwrap # help wrap text for single line output 
display(Markdown(textwrap.fill(response.output_text,width=120)))


Hello, fabulous reader! 🌈✨ Get ready to be whisked away on a delightful adventure filled with intrigue, quirky
characters, and a splash of transcendent reasoning as we delve into *The Innocence of Father Brown* by G.K. Chesterton.
A delightful tale, where crime meets charm and intellect takes on trickery, you’re bound to enjoy this whimsical
escapade!  In our shimmering narrative, we’re introduced to the indomitable *Monsieur Aristide Valentin*, a sharp-witted
detective, hot on the trail of an infamous British criminal named *Flambeau*. Our story kicks off with Valentin covertly
observing passengers at Harwich, where, amusingly, he encounters an utterly unremarkable little priest named *Father
Brown*—a character so ordinary he almost blends into the scenery! Yet, as fate would have it, Father Brown proves to be
the unexpected lynchpin in this strange tale of detective work. As Valentin searches high and low, trying to deduce the
whereabouts of the towering Flambeau, the stage starts filling with zany antics like thrown soup and swapped labels that
keep both the readers and our detective guessing!  As the plot unfolds, a series of curious peculiars—like sugar in salt
shakers—lead Valentin into locations where Flambeau very cleverly maneuvers about, evading the detective while aiming
for the targeted sapphire-encrusted cross borne by none other than Father Brown! It is a mad dash filled with moments of
hilarious folly, like the implausibility of finding 'reasonable' connections in wildly unreasonable behaviors.
Ultimately, the cat-and-mouse chase culminates in a meet-cute where Father Brown, with a comically bewildering array of
casual observations, shocks Flambeau by disclosing he is, in fact, three steps ahead all along, revealing the jewel has
safely reached its destination, thanks to Brown's connivingly simple wisdom!  What a decadent tale! 🌟 *The Innocence of
Father Brown* is a whimsical exploration of intellect over sheer bravado—reminding us that even those seeming
unremarkable amongst us can possess insights sharper than a thief’s dagger! Father Brown emerges as a charming hero
whose ability to discern truth from simple reasons—or even the lack thereof—is appetizingly refreshing. This book is a
delightful reminder that reasoning, laughter, and adroit observation can triumph over misty motivations, making it an
absolute treat for mystery lovers!

# Answer Relevancy

The answer relevancy metric evaluates how relevant the actual output of the LLM app is compared to the provided input. This metric is self-explaining in the sense that the output includes a reason for the metric score.

The metric is calculated as:

$$
AnswerRelevancy=\frac{NumberRelevantStatements}{TotalStatements}
$$

Reference: [Answer Relevancy](https://deepeval.com/docs/metrics-answer-relevancy). 

In [6]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text,
    
)

## run the test
metric.measure(test_case)

Output()

1.0

In [7]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 1.0

**Reason**: The score is 1.00 because the response directly addressed the request for a summary of the story, including key characters and plot points, without any irrelevant statements. The output was concise and relevant, fulfilling the requirements of the input.

# Other Metrics

Other useful metric functions include:

+ [Faithfulness](https://deepeval.com/docs/metrics-faithfulness): evaluates whether the `actual_output` factually aligns with the contents of  `retrieval_context`. 
+ [Contextual Precision](https://deepeval.com/docs/metrics-contextual-precision): evaluates whether nodes in your `retrieval_context` that are relevant to the given input are ranked higher than irrelevant ones. 
+ [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall): evaluates the extent of which the retrieval_context aligns with the expected_output. 
+ [Contextual Relevancy](https://deepeval.com/docs/metrics-contextual-relevancy): evaluates the overall relevance of the information presented in your retrieval_context for a given input. 

# G-Eval

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval offers.

In [18]:
instructions = "You are an helpful assistant that specializes in works of fiction."
PROMPT = """
    Based on the story below, answer the question provided.
    <story>
    {story}
    </story>
    <question>
    Who is the main antagonist in the story and what motivates their actions?
    </question>
"""

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=0.7
)



In [20]:
display(Markdown(textwrap.fill(response.output_text,width=120)))

The main antagonist in "The Blue Cross" is Flambeau, a notorious criminal known for his clever and elaborate thefts. His
motivations stem from a combination of his desire for wealth and the thrill of engaging in clever criminal schemes.
Flambeau is depicted as an intelligent and physically imposing figure who enjoys the challenge of outsmarting others,
including law enforcement. In this story, he is particularly motivated by the opportunity to steal a valuable silver
cross with sapphires that Father Brown, a seemingly simple priest, is transporting to a religious congress. Flambeau’s
actions are driven by both greed and a sense of pride in his cunning, which ultimately leads him to underestimate Father
Brown and the detective, Valentin.

## Evaluation Criteria

The most straightforward way to establish a metric is by using a single criteria.

In [8]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [9]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.7805664599910848, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response effectively summarizes the story while capturing key characters like Monsieur Aristide Valentin and Father Brown, as well as important plot points such as the chase for Flambeau and the clever twists involving the sapphire cross. The introduction and conclusion add a personal touch, enhancing reader engagement. However, the summary could be more concise, as it slightly exceeds the four-paragraph limit specified in the prompt., error: None)

For test case:

  - input: 
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    The Project Gutenberg eBook of The innocence of Father Brown

This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it 

⚠ WARNING: No hyperparameters logged.
» ]8;id=803137;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.91s | token cost: 0.00177495 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.7805664599910848, reason='The response effectively summarizes the story while capturing key characters like Monsieur Aristide Valentin and Father Brown, as well as important plot points such as the chase for Flambeau and the clever twists involving the sapphire cross. The introduction and conclusion add a personal touch, enhancing reader engagement. However, the summary could be more concise, as it slightly exceeds the four-paragraph limit specified in the prompt.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00177495, verbose_logs='Criteria:\nDetermine whether the actual output is factually correct based on the context. \n \nEvaluation Steps:\n[\n    "Identify the context of the input to understand the expected output.",\n    "Compare the actual output against the factual information r

## Evaluation Steps 

G-Eval is flexible in many ways: notice that we can establish an evaluation criteria or a set of evaluation steps, that can help in guiding the model to follow specific steps to perform the evaluation.

In [23]:
...

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'input'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [24]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
result = evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.8325631368251342, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response accurately identifies Flambeau as the main antagonist and provides a clear explanation of his motivations, including greed and the thrill of outsmarting others. However, it could benefit from more detail regarding the specific methods Flambeau employs in his criminal activities and how they relate to the plot, which would enhance the depth of the analysis., error: None)

For test case:

  - input: 
    Based on the story below, answer the question provided.
    <story>
    The Project Gutenberg eBook of The innocence of Father Brown

This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. 

⚠ WARNING: No hyperparameters logged.
» ]8;id=691220;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.69s | token cost: 0.0016501499999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [25]:
result.model_dump()

{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'Correctness [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.8325631368251342,
     'reason': 'The response accurately identifies Flambeau as the main antagonist and provides a clear explanation of his motivations, including greed and the thrill of outsmarting others. However, it could benefit from more detail regarding the specific methods Flambeau employs in his criminal activities and how they relate to the plot, which would enhance the depth of the analysis.',
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.0016501499999999998,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the facts in \'actual output\' contradicts any facts in \'input\'",\n    "You should also heavily penalize omission of detail",\n    "Vague language, or contradicting OPINIONS, are not OK"\n] \n \nRu